# COT_lab — Results

Loads `outputs/eval_results/` and renders the final accuracy table, ReCEval summary, and per-condition violin/bar plots. Run after Stage 5 has produced its CSVs.

In [ ]:
from pathlib import Path
import json
import math

import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path('..').resolve()
EVAL = REPO_ROOT / 'outputs' / 'eval_results'
PLOTS = REPO_ROOT / 'outputs' / 'plots'
print('eval dir:', EVAL)

## 1. Accuracy

Reference (Ho et al. 2023, FLAN-T5-base on GSM8K): baseline 2.50%, standard FT 5.08%, CoT FT 4.40%.

In [ ]:
acc = pd.read_csv(EVAL / 'accuracy.csv')
acc['accuracy_pct'] = (acc['accuracy'] * 100).round(2)
acc

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(acc['condition'], acc['accuracy_pct'], color='steelblue')
ax.set_xticklabels(acc['condition'], rotation=25, ha='right')
ax.set_ylabel('accuracy (%)')
ax.set_title('GSM8K test accuracy by condition')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## 2. ReCEval summary

In [ ]:
rec = pd.read_csv(EVAL / 'receval_summary.csv')
rec

In [ ]:
# Per-example distributions
by_cond = {}
for p in sorted(EVAL.glob('*_receval.jsonl')):
    cond = p.stem.replace('_receval', '')
    with p.open() as f:
        by_cond[cond] = [json.loads(l) for l in f if l.strip()]
list(by_cond.keys())

In [ ]:
metrics = ['intra', 'inter', 'info']
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric in zip(axes, metrics):
    data = [[r[metric] for r in by_cond[c] if not math.isnan(r.get(metric, float('nan')))] for c in by_cond]
    labels = list(by_cond.keys())
    ax.violinplot(data, showmedians=True)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
    ax.set_title(metric)
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('ReCEval distributions by condition')
fig.tight_layout()
plt.show()

## 3. Accuracy x ReCEval — "right answer, wrong reasoning?"

For each condition, split per-example ReCEval scores by whether the predicted answer is correct.

In [ ]:
def is_correct(row):
    p, g = row.get('parsed_answer'), row.get('gold_answer')
    if p is None or g is None:
        return False
    try:
        return abs(float(p) - float(g)) < 1e-6
    except (TypeError, ValueError):
        return False

rows = []
for cond, recs in by_cond.items():
    for r in recs:
        rows.append({
            'condition': cond,
            'correct': is_correct(r),
            'intra': r.get('intra'),
            'inter': r.get('inter'),
            'info': r.get('info'),
        })
df = pd.DataFrame(rows)
df.groupby(['condition', 'correct'])[metrics].mean().round(3)